In [6]:
from pydantic import BaseModel,Field
from typing import Literal,List


In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
class ImageSpec(BaseModel):
    placeholder:str=Field(...,description="e.g. [[IMAGE_1]]")
    filename:str=Field(...,description="Save under images/, e.g. qkv_flow.png")
    prompt:str=Field(...,description="Prompt to send to the image model")
    size:Literal["1024x1024","1024x1536","1536x1024"]="1025x1024"
    quality: Literal["low", "medium", "high"] = "medium"


class GlobalImagePlan(BaseModel):
    md_with_placeholders:str
    images:List[ImageSpec]=Field(default_factory=list)

In [8]:
from langchain_aws import ChatBedrockConverse


In [9]:
LLM_MODEL_ID = "us.meta.llama3-3-70b-instruct-v1:0"
LLM_REGION = "us-east-1"
llm = ChatBedrockConverse(
    model_id=LLM_MODEL_ID,
    region_name=LLM_REGION
)

In [11]:
placehonder="""You are an expert technical blog image planning assistant.

Your job is to analyze a Markdown blog post and generate a structured image plan.

You MUST return output strictly matching the Pydantic model `GlobalImagePlan`.

-----------------------------------------
YOUR TASK
-----------------------------------------

You will receive a Markdown blog as input.

You must:

1. Keep the Markdown EXACTLY the same.
2. DO NOT rewrite, summarize, improve, or modify any text.
3. DO NOT remove or change any formatting.
4. Only insert image placeholders where images would improve clarity.

-----------------------------------------
WHERE TO INSERT IMAGES
-----------------------------------------

Insert placeholders only:
- After major section headings (## or ###)
- After complex explanations
- After architecture descriptions
- After workflows
- After comparisons
- Where diagrams would help understanding
- Where visual examples would add clarity

DO NOT:
- Add images randomly
- Add too many images
- Break code blocks
- Insert placeholders inside code blocks
- Modify existing content

-----------------------------------------
PLACEHOLDER FORMAT
-----------------------------------------

Use this exact format:

[[IMAGE_1]]
[[IMAGE_2]]
[[IMAGE_3]]

Number them sequentially.

-----------------------------------------
IMAGE SPEC RULES
-----------------------------------------

For each placeholder generate an ImageSpec with:

- placeholder: exact placeholder string (e.g. [[IMAGE_1]])
- filename: save under images/ directory (example: images/attention_flow.png)
- prompt: highly detailed image generation prompt describing what the image should show
- size: choose one of:
    - 1024x1024 (for square diagrams)
    - 1536x1024 (for wide architecture diagrams)
    - 1024x1536 (for vertical infographics)
- quality: "medium" unless diagram is complex → use "high"

The prompt must:
- Be descriptive
- Mention diagram style
- Mention labels
- Mention arrows and flow
- Mention clean white background
- Mention professional technical illustration style

-----------------------------------------
IMPORTANT OUTPUT RULES
-----------------------------------------

You MUST return ONLY a valid GlobalImagePlan JSON object.

Do NOT include:
- Explanations
- Extra text
- Markdown fences
- Comments
- Any text before or after the JSON

-----------------------------------------
OUTPUT FORMAT
-----------------------------------------

{
  "md_with_placeholders": "...full markdown with inserted placeholders...",
  "images": [
    {
      "placeholder": "[[IMAGE_1]]",
      "filename": "images/example.png",
      "prompt": "Detailed image generation prompt...",
      "size": "1536x1024",
      "quality": "medium"
    }
  ]
}"""

In [14]:
from langchain.messages import SystemMessage,HumanMessage

In [16]:
markdown="""
# State of Multimodal LLMs in 2026

## Introduction to Multimodal LLMs
Recent developments in multimodal LLMs have shown significant progress, with models now capable of processing and generating multiple forms of data, such as text, images, and audio [Not found in provided sources]. 
* Multimodal LLMs have been applied to various tasks, including visual question answering, image captioning, and text-to-image synthesis.
* The impact of multimodal LLMs can be seen in industries like healthcare, education, and entertainment, where they are used for applications such as medical image analysis, interactive learning systems, and content creation [Not found in provided sources].
* Despite the advancements, key challenges in multimodal LLM research remain, including the need for large-scale datasets, improved model architectures, and better evaluation metrics [Not found in provided sources].

## Recent Advances in Multimodal LLMs
Recent breakthroughs in multimodal LLM architecture have led to significant improvements in the field. 
* Multimodal transformers, which combine visual and textual features, have shown promising results in tasks such as visual question answering and image-text retrieval [Not found in provided sources].
* The use of multimodal attention mechanisms has also been explored, allowing models to focus on specific parts of the input data [Not found in provided sources].

Multimodal LLMs play a crucial role in both computer vision and natural language processing. 
They can be used to analyze and understand visual data, such as images and videos, and generate text-based descriptions or summaries.
In natural language processing, multimodal LLMs can be used to improve language understanding and generation tasks, such as machine translation and text summarization.

The potential applications of multimodal LLMs in healthcare are vast. 
They can be used to analyze medical images, such as X-rays and MRIs, and generate text-based diagnoses or recommendations.
Additionally, multimodal LLMs can be used to develop personalized treatment plans and improve patient outcomes [Not found in provided sources].
Overall, the latest advancements in multimodal LLMs have the potential to revolutionize various fields, including healthcare, and improve the way we interact with and understand visual and textual data.

## Challenges and Limitations
The development of multimodal LLMs has made significant progress, but there are still several challenges and limitations that need to be addressed. 
* The limitations of current multimodal LLM models include their inability to fully understand the nuances of human communication, such as sarcasm, idioms, and figurative language [Not found in provided sources].
* Training and deploying multimodal LLMs pose significant challenges, including the need for large amounts of diverse and high-quality training data, as well as the requirement for significant computational resources [Not found in provided sources].
* Further research is needed to improve the performance and robustness of multimodal LLMs, particularly in areas such as common sense reasoning, emotional intelligence, and adaptability to new contexts and domains [Not found in provided sources]. 
Overall, addressing these challenges and limitations will be crucial to unlocking the full potential of multimodal LLMs and achieving more effective and engaging human-computer interactions.

## Future Directions
The future of multimodal LLMs holds great promise, with potential applications in areas such as [virtual assistants](Not found in provided sources) and [human-computer interaction](Not found in provided sources). 
* Multimodal LLMs may be used to improve accessibility and user experience in various domains.
* The role of multimodal LLMs in shaping the future of AI is significant, as they can enable more natural and intuitive interactions between humans and machines.
* Continued research in multimodal LLMs is crucial to overcome current limitations and unlock their full potential, driving innovation and progress in the field of AI [Not found in provided sources].

"""

In [18]:
output=llm.with_structured_output(GlobalImagePlan)\
.invoke(
    [
        SystemMessage(content=placehonder),
        HumanMessage(content=markdown)
    ]
)

output

GlobalImagePlan(md_with_placeholders='# State of Multimodal LLMs in 2026\n## Introduction to Multimodal LLMs\nRecent developments in multimodal LLMs have shown significant progress, with models now capable of processing and generating multiple forms of data, such as text, images, and audio [Not found in provided sources]. \n* Multimodal LLMs have been applied to various tasks, including visual question answering, image captioning, and text-to-image synthesis.\n* The impact of multimodal LLMs can be seen in industries like healthcare, education, and entertainment, where they are used for applications such as medical image analysis, interactive learning systems, and content creation [Not found in provided sources].\n* Despite the advancements, key challenges in multimodal LLM research remain, including the need for large-scale datasets, improved model architectures, and better evaluation metrics [Not found in provided sources].\n[[IMAGE_1]]\n## Recent Advances in Multimodal LLMs\nRecent 

In [20]:
output.md_with_placeholders

'# State of Multimodal LLMs in 2026\n## Introduction to Multimodal LLMs\nRecent developments in multimodal LLMs have shown significant progress, with models now capable of processing and generating multiple forms of data, such as text, images, and audio [Not found in provided sources]. \n* Multimodal LLMs have been applied to various tasks, including visual question answering, image captioning, and text-to-image synthesis.\n* The impact of multimodal LLMs can be seen in industries like healthcare, education, and entertainment, where they are used for applications such as medical image analysis, interactive learning systems, and content creation [Not found in provided sources].\n* Despite the advancements, key challenges in multimodal LLM research remain, including the need for large-scale datasets, improved model architectures, and better evaluation metrics [Not found in provided sources].\n[[IMAGE_1]]\n## Recent Advances in Multimodal LLMs\nRecent breakthroughs in multimodal LLM archi

In [21]:
output.images

[ImageSpec(placeholder='[[IMAGE_1]]', filename='images/multimodal_llm_architecture.png', prompt='A diagram showing the architecture of a multimodal LLM, with visual and textual features combined, and labels and arrows indicating the flow of data, on a clean white background, in a professional technical illustration style', size='1536x1024', quality='medium'),
 ImageSpec(placeholder='[[IMAGE_2]]', filename='images/multimodal_transformers.png', prompt='An illustration of multimodal transformers, with visual and textual features combined, and labels and arrows indicating the flow of data, on a clean white background, in a professional technical illustration style', size='1024x1024', quality='medium'),
 ImageSpec(placeholder='[[IMAGE_3]]', filename='images/multimodal_llm_applications.png', prompt='A diagram showing the various applications of multimodal LLMs, including computer vision and natural language processing, with labels and arrows indicating the relationships between the different

In [23]:
output.images[0].prompt

'A diagram showing the architecture of a multimodal LLM, with visual and textual features combined, and labels and arrows indicating the flow of data, on a clean white background, in a professional technical illustration style'